# Lab 4: Data Wrangling with Pandas

**Course:** Data Analytics Laboratory  
**Contributors:** Abdul Ahad, Abdullah Jabbar  

---

## Objectives
- Load data from CSV, Excel, JSON files
- Explore data: head(), tail(), info(), describe(), shape
- Select columns and filter rows using Boolean indexing
- Sort data by single and multiple columns
- Handle missing data: isnull(), dropna(), fillna()
- Add, remove, and rename columns

**Dataset:** Titanic Dataset

## 4.1 — Loading Data from Multiple Formats

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Create a realistic Titanic-like dataset
np.random.seed(42)
n = 891

# Generate data
pclass = np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55])
sex = np.random.choice(['male', 'female'], n, p=[0.65, 0.35])
age = np.random.normal(30, 14, n).clip(0.5, 80).round(1)
sibsp = np.random.choice(range(9), n, p=[0.68, 0.23, 0.05, 0.02, 0.01, 0.003, 0.003, 0.002, 0.002])
parch = np.random.choice(range(7), n, p=[0.76, 0.12, 0.08, 0.02, 0.01, 0.005, 0.005])

# Fare based on class
fare = np.where(pclass == 1, np.random.exponential(80, n) + 30,
       np.where(pclass == 2, np.random.exponential(20, n) + 10,
                np.random.exponential(8, n) + 5))
fare = fare.round(4)

# Survival rates influenced by class and sex
survival_prob = np.where((sex == 'female') & (pclass == 1), 0.97,
                np.where((sex == 'female') & (pclass == 2), 0.92,
                np.where((sex == 'female') & (pclass == 3), 0.50,
                np.where((sex == 'male') & (pclass == 1), 0.37,
                np.where((sex == 'male') & (pclass == 2), 0.16, 0.14)))))
survived = np.random.binomial(1, survival_prob)

embarked = np.random.choice(['S', 'C', 'Q'], n, p=[0.72, 0.19, 0.09])

titanic = pd.DataFrame({
    'PassengerId': range(1, n + 1),
    'Survived': survived,
    'Pclass': pclass,
    'Name': [f"Passenger_{i}" for i in range(1, n + 1)],
    'Sex': sex,
    'Age': age,
    'SibSp': sibsp,
    'Parch': parch,
    'Fare': fare,
    'Embarked': embarked
})

# Introduce realistic missing values
age_mask = np.random.random(n) < 0.20  # ~20% missing ages
titanic.loc[age_mask, 'Age'] = np.nan
emb_mask = np.random.random(n) < 0.02  # ~2% missing embarked
titanic.loc[emb_mask, 'Embarked'] = np.nan

# Save in multiple formats
titanic.to_csv('titanic.csv', index=False)
titanic.to_json('titanic.json', orient='records', indent=2)
titanic.head(100).to_excel('titanic_sample.xlsx', index=False)

print("Datasets created:")
print("  ✓ titanic.csv")
print("  ✓ titanic.json")
print("  ✓ titanic_sample.xlsx")

In [ ]:
# Load from different formats
df_csv = pd.read_csv('titanic.csv')
df_json = pd.read_json('titanic.json')
df_excel = pd.read_excel('titanic_sample.xlsx')

print(f"CSV:   {df_csv.shape[0]} rows x {df_csv.shape[1]} cols")
print(f"JSON:  {df_json.shape[0]} rows x {df_json.shape[1]} cols")
print(f"Excel: {df_excel.shape[0]} rows x {df_excel.shape[1]} cols")

# Use CSV as primary dataset
df = df_csv.copy()
print(f"\nUsing CSV dataset: {df.shape}")

## 4.2 — Data Exploration

In [ ]:
# First few rows
print("=== head() ===")
df.head()

In [ ]:
# Last few rows
print("=== tail() ===")
df.tail()

In [ ]:
# Dataset info
print("=== info() ===")
df.info()

In [ ]:
# Statistical summary
print("=== describe() ===")
df.describe()

In [ ]:
# Additional exploration
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Data types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nMissing percentages:\n{(df.isnull().sum() / len(df) * 100).round(2)}%")
print(f"\nUnique values per column:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()}")

## 4.3 — Column Selection & Row Filtering

In [ ]:
# Select single column
print("=== Single Column ===")
print(df['Age'].head(10))

# Select multiple columns
subset = df[['Name', 'Sex', 'Age', 'Survived']]
print(f"\n=== Multiple Columns ===")
print(subset.head())

# Boolean indexing — filter rows
print("\n=== Filtering with Boolean Indexing ===")

# Survived passengers
survived = df[df['Survived'] == 1]
print(f"Survived: {len(survived)} passengers")

# First class females
first_class_female = df[(df['Pclass'] == 1) & (df['Sex'] == 'female')]
print(f"1st class females: {len(first_class_female)}")

# Age > 50
seniors = df[df['Age'] > 50]
print(f"Passengers over 50: {len(seniors)}")

# High fare passengers (top 5%)
fare_threshold = df['Fare'].quantile(0.95)
high_fare = df[df['Fare'] > fare_threshold]
print(f"Top 5% fare (> {fare_threshold:.2f}): {len(high_fare)} passengers")

# Complex filter: survived males from 3rd class
survived_3rd_male = df[(df['Survived'] == 1) & (df['Pclass'] == 3) & (df['Sex'] == 'male')]
print(f"Survived 3rd class males: {len(survived_3rd_male)}")

# Using .query() method
young_survivors = df.query("Age < 18 and Survived == 1")
print(f"Survived children (< 18): {len(young_survivors)}")

## 4.4 — Sorting

In [ ]:
# Sort by single column
print("=== Sort by Age (ascending) ===")
print(df.sort_values('Age').head())

print("\n=== Sort by Fare (descending) — Top 10 ===")
print(df.sort_values('Fare', ascending=False)[['Name', 'Pclass', 'Fare']].head(10))

# Sort by multiple columns
print("\n=== Sort by Pclass (asc) then Fare (desc) ===")
multi_sort = df.sort_values(['Pclass', 'Fare'], ascending=[True, False])
print(multi_sort[['Name', 'Pclass', 'Fare']].head(10))

# Rank passengers by fare within each class
df['Fare_Rank'] = df.groupby('Pclass')['Fare'].rank(ascending=False).astype(int)
print("\n=== Top fare passenger per class ===")
top_per_class = df.sort_values('Fare_Rank').groupby('Pclass').first()
print(top_per_class[['Name', 'Fare', 'Fare_Rank']])

## 4.5 — Handling Missing Data

In [ ]:
# Check missing data
print("=== Missing Data Summary ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Missing'] > 0])

# Visualize missing pattern
print(f"\nRows with any missing: {df.isnull().any(axis=1).sum()}")
print(f"Complete rows: {df.dropna().shape[0]}")

In [ ]:
# Different strategies for handling missing values
df_clean = df.copy()

# Strategy 1: Fill Age with median by class
print("=== Strategy 1: Fill Age by Class Median ===")
for pclass in [1, 2, 3]:
    median_age = df_clean.loc[df_clean['Pclass'] == pclass, 'Age'].median()
    mask = (df_clean['Pclass'] == pclass) & (df_clean['Age'].isnull())
    df_clean.loc[mask, 'Age'] = median_age
    print(f"  Class {pclass} median age: {median_age:.1f}")

# Strategy 2: Fill Embarked with mode
mode_embarked = df_clean['Embarked'].mode()[0]
df_clean['Embarked'].fillna(mode_embarked, inplace=True)
print(f"\nEmbarked filled with mode: '{mode_embarked}'")

# Verify
print(f"\n=== After Cleaning ===")
print(f"Missing values remaining: {df_clean.isnull().sum().sum()}")
print(f"Dataset shape: {df_clean.shape}")

# Demo: dropna
dropped = df.dropna(subset=['Age'])
print(f"\ndropna(subset=['Age']): {len(df)} → {len(dropped)} rows")

## 4.6 — Add, Remove & Rename Columns

In [ ]:
# Add new columns
df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1
df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int)
df_clean['AgeGroup'] = pd.cut(df_clean['Age'], bins=[0, 12, 18, 35, 60, 100],
                               labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior'])
df_clean['FarePerPerson'] = df_clean['Fare'] / df_clean['FamilySize']

print("=== New Columns Added ===")
print(df_clean[['Name', 'FamilySize', 'IsAlone', 'AgeGroup', 'FarePerPerson']].head(10))

# Rename columns
df_renamed = df_clean.rename(columns={
    'Pclass': 'TicketClass',
    'SibSp': 'SiblingsSpouses',
    'Parch': 'ParentsChildren'
})
print(f"\nRenamed columns: {list(df_renamed.columns)}")

# Remove columns
df_final = df_clean.drop(columns=['Fare_Rank'])
print(f"\nAfter dropping 'Fare_Rank': {list(df_final.columns)}")

# GroupBy analysis
print("\n=== Survival Rate by Class and Sex ===")
survival_analysis = df_clean.groupby(['Pclass', 'Sex'])['Survived'].agg(['mean', 'count'])
survival_analysis.columns = ['Survival Rate', 'Count']
survival_analysis['Survival Rate'] = (survival_analysis['Survival Rate'] * 100).round(1)
print(survival_analysis)

In [ ]:
# Save cleaned dataset
df_clean.to_csv('titanic_cleaned.csv', index=False)
print(f"Cleaned dataset saved: titanic_cleaned.csv ({df_clean.shape[0]} rows, {df_clean.shape[1]} cols)")

# Final summary
print(f"\n=== Data Cleaning Summary ===")
print(f"Original shape: {df.shape}")
print(f"Cleaned shape:  {df_clean.shape}")
print(f"Missing values before: {df.isnull().sum().sum()}")
print(f"Missing values after:  {df_clean.isnull().sum().sum()}")
print(f"New columns added: FamilySize, IsAlone, AgeGroup, FarePerPerson")

---
### Summary

| Task | Status |
|---|---|
| Load data from CSV, Excel, JSON | Done |
| Data exploration (head, tail, info, describe) | Done |
| Column selection & Boolean filtering | Done |
| Sorting (single & multiple columns) | Done |
| Missing data handling | Done |
| Add, remove, rename columns | Done |

---
*Lab 4 completed successfully.*